In [2]:
import numpy as np
import pandas as pd

# ============================================================
# INPUT FILE
# ============================================================
DES_FILE = "/Users/aishwarya/Downloads/des.csv"

# ============================================================
# DECam-equivalent limiting magnitudes (from YOUR analysis)
# ============================================================
Y_LIMIT = 24.87   # Y-band 4σ
Z_LIMIT = 25.79   # z-band 4σ

# ============================================================
# LOAD DES CATALOG
# ============================================================
des = pd.read_csv(DES_FILE)

# ============================================================
# REQUIRED DES COLUMNS
# ============================================================
required_cols = [
    "ra", "dec",
    "mag_auto_i", "mag_auto_z", "mag_auto_y",
    "magerr_auto_i", "magerr_auto_z", "magerr_auto_y"
]

missing = [c for c in required_cols if c not in des.columns]
if missing:
    raise RuntimeError(f"Missing DES columns: {missing}")

# ============================================================
# MAGERR → SNR (IDENTICAL TO DECam)
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

des["i_snr"] = magerr_to_snr(des["magerr_auto_i"])
des["z_snr"] = magerr_to_snr(des["magerr_auto_z"])
des["y_snr"] = magerr_to_snr(des["magerr_auto_y"])

# ============================================================
# TRUNCATE DES TO DECam DEPTH (CRUCIAL)
# ============================================================
des = des[
    (des["mag_auto_y"] < Y_LIMIT) &
    (des["mag_auto_z"] < Z_LIMIT)
].copy()

print(f"DES sources after depth matching: {len(des)}")

# ============================================================
# COLORS + ERRORS
# ============================================================
des["z_y"] = des["mag_auto_z"] - des["mag_auto_y"]
des["i_z"] = des["mag_auto_i"] - des["mag_auto_z"]

des["z_y_err"] = np.sqrt(
    des["magerr_auto_z"]**2 +
    des["magerr_auto_y"]**2
)

# ============================================================
# DROP-OUT LOGIC (MATCHES DECam)
# ============================================================
cut_y_detected = des["y_snr"] > 3.0
cut_z_detected = des["z_snr"] > 3.0

cut_i_dropout  = des["i_snr"] < 2.0

cut_zy_color   = des["z_y"] > 0.8
cut_iz_break   = des["i_z"] > 1.0

cut_sig_color  = des["z_y"] > 2.5 * des["z_y_err"]

# ============================================================
# FINAL DES LBG SELECTION
# ============================================================
final_sel = (
    cut_y_detected &
    cut_z_detected &
    cut_i_dropout &
    cut_zy_color &
    cut_iz_break &
    cut_sig_color
)

des_lbg = des[final_sel].copy()

print(f"Final DES LBGs: {len(des_lbg)}")

# ============================================================
# OUTPUT CATALOG
# ============================================================
out_file = "/Users/aishwarya/Documents/Lyman_alpha/DES_LBG_equivalent.cat"

des_lbg[[
    "ra", "dec",
    "mag_auto_y", "mag_auto_z", "mag_auto_i",
    "y_snr", "z_snr", "i_snr",
    "z_y", "i_z"
]].to_csv(out_file, index=False)

print(f"Saved DES-equivalent LBG catalog to:\n{out_file}")


DES sources after depth matching: 499994
Final DES LBGs: 12
Saved DES-equivalent LBG catalog to:
/Users/aishwarya/Documents/Lyman_alpha/DES_LBG_equivalent.cat
